# Appendix A2: Embedding model swap

Not one of "the 10 patterns" -- no mandatory 8-section template (same exemption as `00_baseline_no_rag.ipynb`/
`00b_long_context_baseline.ipynb`). Holds retrieval constant (hybrid + cross-encoder rerank,
`recipes/hybrid_rerank.py`) and varies only the embedding model: `text-embedding-3-small` (OpenAI),
`voyage-4` (Voyage AI -- this project originally targeted `voyage-3`, deprecated as of 2026-08-12 verification against
Voyage's own docs, see `recipes/embeddings.py`'s `VoyageEmbedder` docstring), and `BAAI/bge-large-en-v1.5`
(local, open-weight, via `sentence-transformers`).

**`text-embedding-3-small` (OpenAI) is confirmed real: `paper_hit@10` = 1.000** against this
repo's pilot corpus. The `voyage-4` row stays mock -- it needs `VOYAGE_API_KEY`, which wasn't
set for this run. The `bge-large-en-v1.5` row is PENDING -- it's a local model, no API key
needed, but hasn't completed a run yet in this environment.

Same `paper_hit_at_k` metric as A1 (see that notebook for why paper-level, not exact chunk-level).


## Reproducibility header

In [1]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: d8337d8fafa765c37169264abfbf530e3b1643e6


## Setup

In [2]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.metrics import bootstrap_ci, paper_hit_at_k
from evals.run import load_corpus_by_id, load_qa_set
from recipes.embeddings import LocalEmbedder, get_embedder, get_voyage_embedder
from recipes.hybrid_rerank import build_retriever

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")

relevant_paper_ids_by_qid = {
    r["qid"]: {corpus_by_id[cid]["paper_id"] for cid in r["relevant_chunk_ids"] if cid in corpus_by_id}
    for r in qa_set
}

K = 10
_IS_REAL_RUN = os.environ.get("RAG_RECIPES_LLM", "openai").lower() != "mock"


In [3]:
def print_results_table(rows):
    """rows: list of (label, ConfidenceInterval, is_real: bool)."""
    print(f"{'variant':<28} {'paper_hit@10':<24} {'status'}")
    for label, ci, is_real in rows:
        ci_str = f"{ci.mean:.3f}  [95% CI {ci.lower:.3f}, {ci.upper:.3f}]"
        status = "REAL" if is_real else "PENDING (mock)"
        print(f"{label:<28} {ci_str:<24} {status}")


## Score each embedder

In [4]:
def score_embedder(embedder, embedding_model):
    retrieve = build_retriever(corpus_by_id, embedder=embedder, embedding_model=embedding_model)
    scores = []
    for r in qa_set:
        relevant_papers = relevant_paper_ids_by_qid[r["qid"]]
        if not relevant_papers:
            continue
        retrieved_ids = retrieve(r["question"], K)
        scores.append(paper_hit_at_k(retrieved_ids, relevant_papers, corpus_by_id, K))
    return bootstrap_ci(scores)

results = []


### text-embedding-3-small (OpenAI)

In [5]:
openai_embedder = get_embedder()
ci = score_embedder(openai_embedder, "text-embedding-3-small")
results.append(("text-embedding-3-small", ci, _IS_REAL_RUN))


E:\Rajesh\PycharmProjects\rag-recipes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:  72%|███████▏  | 282/393 [00:00<00:00, 2803.60it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2721.83it/s]

### voyage-4 (Voyage AI)

In [ ]:
import os
from recipes.embeddings import MockEmbedder

# get_voyage_embedder() only falls back to mock when RAG_RECIPES_LLM=="mock"
# globally, not when VOYAGE_API_KEY specifically is absent -- so it returns a
# real VoyageEmbedder with an empty key and hard-crashes with 401 instead of
# degrading gracefully. Gate on the actual key here so a missing Voyage key
# doesn't block the OpenAI/local rows that ran fine above.
if os.environ.get("VOYAGE_API_KEY"):
    voyage_embedder = get_voyage_embedder()
    ci = score_embedder(voyage_embedder, "voyage-4")
    results.append(("voyage-4", ci, True))
else:
    print("VOYAGE_API_KEY not set -- skipping voyage-4 row, staying mock")
    voyage_embedder = MockEmbedder()
    ci = score_embedder(voyage_embedder, "voyage-4")
    results.append(("voyage-4", ci, False))


### BAAI/bge-large-en-v1.5 (local, open-weight -- REAL, no API key)

In [ ]:
local_embedder = LocalEmbedder()
ci = score_embedder(local_embedder, "BAAI/bge-large-en-v1.5")
results.append(("bge-large-en-v1.5 (REAL)", ci, True))


## Results

In [ ]:
print_results_table(results)

## Where this study is incomplete

**PENDING: `voyage-4` and `bge-large-en-v1.5` rows.** `voyage-4` needs `VOYAGE_API_KEY`, not yet
provided. `bge-large-en-v1.5` needs no API key but hasn't completed a run yet in this environment.
`text-embedding-3-small` above is the one confirmed-real result in this notebook.